# Skills → Title: the Reverse Model (M18)

The production classifier (`/cv/role`) predicts a role from **raw CV text**
(TF-IDF + LogReg). The team's 05/07 decision called for the reverse direction —
predict the role from the **extracted skill set** (*occupation prediction from
skill profiles / bidirectional role-skill inference*). This notebook builds that
model and measures it honestly:

1. Train on two domains: **job postings** (41.7K ready pairs from the M06 pipeline)
   and **CVs** (11.8K extracted with the same checkpointed SkillNer infrastructure).
2. Internal held-out metrics + cross-domain transfer (domain shift).
3. **The real test:** the 32 authentic CVs (M04 ground truth) through the full
   pipeline (PDF → text → SkillNer → predict).
4. **Head-to-head** against the shipped text classifier on the exact same files +
   agreement analysis (the basis for a cross-validation signal).

Pre-declared primary model (to avoid cherry-picking): **CV-domain Logistic
Regression**. Everything else is reported as context. If the shipped classifier
wins — that is a legitimate, interesting finding, not a failure.

Label space: the same 12 canonical roles covered by the lang-uk corpus (subset of
the 59-title taxonomy). CVs whose true role is outside this space are reported as
a structural coverage limitation, never silently dropped.

In [1]:
import os, sys, json, warnings
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.environ.setdefault('MONGO_URI', 'mongodb://localhost:27017/careerlens')
sys.path.insert(0, os.path.abspath('.'))          # run from ds/model
from pymongo import MongoClient
from taxonomy import CANONICAL_TITLES

client = MongoClient(os.environ['MONGO_URI'])
db = client['careerlens']
JOBS_SKILLS, CV_SKILLS = 'lang-uk-job-skills', 'lang-uk-cv-skills'
print({c: db[c].estimated_document_count() for c in (JOBS_SKILLS, CV_SKILLS)})

def doc_skills(doc):
    """Skill set of one document: SkillNer full matches + high-confidence ngrams."""
    seen = set()
    sk = doc.get('skills') or {}
    for m in sk.get('full_matches', []):
        v = (m.get('doc_node_value') or '').lower().strip()
        if len(v) >= 3: seen.add(v)
    for m in sk.get('ngram_matches', []):
        v = (m.get('doc_node_value') or '').lower().strip()
        if len(v) >= 3 and float(m.get('score', 0)) >= 0.9: seen.add(v)
    return seen

{'lang-uk-job-skills': 41745, 'lang-uk-cv-skills': 11776}


## 1. Training data — two domains

Jobs: the (skills, canonical title) pairs already produced by the M06 extraction —
real postings only (the marked synthetic `augmented-2026` collection is **excluded**;
a synthetic ramp must not teach the reverse model). CVs: extracted from
`lang-uk-cv` with the same parallel SkillNer runner; labels come from the corpus'
own `Primary Keyword` tag through the existing mapping — no noisy title
normalization involved.

In [2]:
def load_domain(coll_name):
    rows = []
    for doc in db[coll_name].find({}, {'og_title': 1, 'skills': 1}):
        s = doc_skills(doc)
        if len(s) >= 3:
            rows.append({'label': doc['og_title'], 'skills': s})
    return pd.DataFrame(rows)

df_jobs = load_domain(JOBS_SKILLS)
df_cvs  = load_domain(CV_SKILLS)
print(f"jobs: {len(df_jobs):,} docs | cvs: {len(df_cvs):,} docs (>=3 skills each)")
pd.DataFrame({'jobs': df_jobs.label.value_counts(),
              'cvs': df_cvs.label.value_counts()}).fillna(0).astype(int)

jobs: 40,532 docs | cvs: 10,679 docs (>=3 skills each)


,jobs,cvs
label,,
Backend Developer,3899,921
C++ Developer,3667,917
Cyber Security,800,676
Data Engineer,2835,954
Data Scientist,2191,947
DevOps Engineer,3922,925
Frontend Developer,3874,897
Java Developer,3894,920
Product Manager,3873,906


## 2. Leakage guard

The shipped classifier's methodology scrubs the declared title string from the
text (measured leakage: 0.981 → 0.932 macro-F1). The analog here: drop any
*skill* that is literally a canonical title name ('data scientist',
'backend developer', …, plural forms included — mirroring `ROLE_NAME_NOISE` in
train.py). Single tech tokens ('java', 'python', 'devops') stay: they are
legitimate skills, and removing them cripples framework-centric roles (the
classifier work measured exactly that). Per-class top features are printed later
so any residual leakage is visible, not hidden.

In [3]:
LEAK = {t.lower() for t in CANONICAL_TITLES} | {t.lower() + 's' for t in CANONICAL_TITLES}
def clean(skills): return {s for s in skills if s not in LEAK}
for df in (df_jobs, df_cvs):
    df['skills'] = df['skills'].map(clean)
removed_jobs = sum(1 for s in df_jobs.skills if not s)
print('leak set size:', len(LEAK), '| docs emptied by guard:', removed_jobs)
df_jobs = df_jobs[df_jobs.skills.map(len) >= 3].reset_index(drop=True)
df_cvs  = df_cvs[df_cvs.skills.map(len) >= 3].reset_index(drop=True)
print(f"after guard - jobs: {len(df_jobs):,} | cvs: {len(df_cvs):,}")

leak set size: 118 | docs emptied by guard: 0
after guard - jobs: 40,529 | cvs: 10,651


## 3. Vectorize + train (multi-hot · LogReg baseline · small MLP)

Multi-hot skill vectors (`min_df=5` — a skill must appear in ≥5 documents to
become a feature). Two models per domain, small grid, stop: this is a research
notebook, not a product.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

def identity(x): return x

def train_domain(df, domain_name):
    vec = CountVectorizer(analyzer=identity, min_df=5, binary=True)
    X = vec.fit_transform(list(df.skills))
    # integer-encoded labels: sklearn's MLP early-stopping scorer calls
    # np.isnan on y_pred, which explodes on string labels
    le = LabelEncoder()
    y = le.fit_transform(df.label.to_numpy(dtype=object))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    out = {}
    for name, model in [
        ('logreg', LogisticRegression(max_iter=2000, C=1.0)),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128,), max_iter=300,
                              early_stopping=True, random_state=42)),
    ]:
        model.fit(Xtr, ytr)
        proba = model.predict_proba(Xte)
        pred = proba.argmax(1)
        top3 = np.argsort(-proba, axis=1)[:, :3]
        top3_acc = np.mean([yt in row for yt, row in zip(yte, top3)])
        out[name] = {'model': model, 'acc': accuracy_score(yte, pred),
                     'top3': top3_acc, 'macro_f1': f1_score(yte, pred, average='macro')}
        print(f"{domain_name}/{name}: acc={out[name]['acc']:.3f} "
              f"top3={top3_acc:.3f} macroF1={out[name]['macro_f1']:.3f} "
              f"({X.shape[1]} skill features)")
    out['vec'], out['le'], out['holdout'] = vec, le, (Xte, yte)
    return out

R_jobs = train_domain(df_jobs, 'jobs')
R_cvs  = train_domain(df_cvs, 'cvs')

jobs/logreg: acc=0.808 top3=0.953 macroF1=0.796 (7362 skill features)


jobs/mlp: acc=0.825 top3=0.962 macroF1=0.814 (7362 skill features)


cvs/logreg: acc=0.793 top3=0.937 macroF1=0.792 (3067 skill features)


cvs/mlp: acc=0.822 top3=0.953 macroF1=0.822 (3067 skill features)


In [5]:
# Cross-domain transfer: does a model trained on postings understand CVs (and vice versa)?
def cross_eval(src, df_other, name):
    Xo = src['vec'].transform(list(df_other.skills))
    m = src['logreg']['model']
    pred = src['le'].inverse_transform(m.predict_proba(Xo).argmax(1))
    y_other = df_other.label.to_numpy(dtype=object)
    acc = accuracy_score(y_other, pred)
    f1 = f1_score(y_other, pred, average='macro')
    print(f"{name}: acc={acc:.3f} macroF1={f1:.3f}")
    return acc, f1

print('--- domain shift ---')
cross_eval(R_jobs, df_cvs, 'trained on JOBS  -> tested on CVs ')
cross_eval(R_cvs, df_jobs, 'trained on CVs   -> tested on JOBS')

--- domain shift ---
trained on JOBS  -> tested on CVs : acc=0.776 macroF1=0.777


trained on CVs   -> tested on JOBS: acc=0.754 macroF1=0.744


(0.7538552641318562, 0.7441640150788097)

In [6]:
# Transparency: top-10 skills per class for the primary model - residual leakage is visible here
m = R_cvs['logreg']['model']; vec = R_cvs['vec']; le = R_cvs['le']
feats = np.array(vec.get_feature_names_out())
rows = []
for i, cls in enumerate(m.classes_):
    top = feats[np.argsort(-m.coef_[i])[:10]]
    rows.append({'role': le.classes_[int(cls)], 'top_skills': ', '.join(top)})
pd.set_option('display.max_colwidth', 130)
pd.DataFrame(rows)

,role,top_skills
0,Backend Developer,"ruby on rail, asp net, c #, laravel, symfony, development testing, woocommerce, scala, rabbitmq, backend"
1,C++ Developer,"c++, cmake, stl, stm32, unreal engine, debugging, rust, threading, linux, circuit design"
2,Cyber Security,"penetration testing, cybersecurity, siem, network security, security policy, kali linux, devsecops, vulnerability, nmap, cisco"
3,Data Engineer,"etl, pl sql, store procedure, ssis, database administrator, data engineering, sql, snowflake, airflow, power bi"
4,Data Scientist,"data science, computer vision, time series, text classification, deep learning, forecasting, machine learn, transformers, powe..."
5,DevOps Engineer,"devops, troubleshooting, system administration, purchasing, technical support, terraform, kubernetes, dns, jenkins, scripting"
6,Frontend Developer,"react, front end, react redux, angular, html5, vue js, react js, vue, redux, sass"
7,Java Developer,"java, spring boot, java 8, hibernate, spring mvc, microservices, integration tests, manual test, maven, spring framework"
8,Product Manager,"project management, product management, managing team, managing teams, scrum, product strategy, managed projects, manager prod..."
9,QA Automation Engineer,"test case, quality assurance, test automation, software testing, postman, bug reports, automation testing, selenium, regressio..."


## 4. The real test — 32 authentic CVs (full pipeline)

PDF → text (PyMuPDF) → SkillNer (local, same stack as the extraction runner) →
multi-hot → predict. Ground truth: `test-fixtures/authentic-cvs/manifest.json`
(`acceptable_titles`; `true_title: "none"` marks non-engineering / reject
fixtures). Coverage honesty: our label space is the 12 lang-uk roles — CVs whose
acceptable titles fall entirely outside it are *structurally out of coverage* for
this model and reported as such.

In [7]:
import fitz  # PyMuPDF

FIXTURES = os.path.abspath(os.path.join('..', '..', 'test-fixtures', 'authentic-cvs'))
manifest = json.load(open(os.path.join(FIXTURES, 'manifest.json'), encoding='utf-8'))['cvs']
print('manifest CVs:', len(manifest))

def pdf_text(path):
    with fitz.open(path) as doc:
        return '\n'.join(page.get_text() for page in doc)

texts = {}
for entry in manifest:
    p = os.path.join(FIXTURES, 'pdfs', entry['file'])
    texts[entry['file']] = pdf_text(p)
print('extracted text from', len(texts), 'PDFs; empty:',
      [f for f, t in texts.items() if len(t.strip()) < 50])

manifest CVs: 32


extracted text from 32 PDFs; empty: ['scanned-image-reject_Old-Format.pdf']


In [8]:
# SkillNer over the 32 texts (local, ~2s/doc)
from extract_skills import build_skill_extractor, extract_skills_from_text
extractor = build_skill_extractor()
cv32_skills = {}
for f, t in texts.items():
    raw = extract_skills_from_text(extractor, t[:20000])
    cv32_skills[f] = clean(doc_skills({'skills': raw}))
print({f: len(s) for f, s in list(cv32_skills.items())[:6]})

loading full_matcher ...


loading abv_matcher ...
loading full_uni_matcher ...


loading low_form_matcher ...


loading token_matcher ...


  SkillNer failed (IndexError): skipping skills for this doc


{'backend-senior-strong_Daniel-Peretz.pdf': 0, 'frontend-mid-strong_Noa-Shapiro.pdf': 26, 'devops-senior-strong_Alex-Vaisman.pdf': 39, 'datascientist-mid-strong_Maya-Berkovich.pdf': 27, 'qa-automation-mid-mid_Yossi-Alfasi.pdf': 31, 'ml-senior-strong_Amir-Dahan.pdf': 28}


In [9]:
COVERED = set(df_cvs.label.unique())
primary = R_cvs['logreg']['model']; pvec = R_cvs['vec']; ple = R_cvs['le']

rows = []
for entry in manifest:
    f = entry['file']
    acceptable = set(entry['acceptable_titles']) if entry['acceptable_titles'] else set()
    in_cov = bool(acceptable & COVERED)
    X = pvec.transform([cv32_skills[f]])
    proba = primary.predict_proba(X)[0]
    order = np.argsort(-proba)
    pred, conf = ple.classes_[order[0]], proba[order[0]]
    top3 = [ple.classes_[i] for i in order[:3]]
    is_none = entry['true_title'] == 'none'
    correct = (pred in acceptable) if (in_cov and not is_none) else None
    rows.append({'file': f, 'true': entry['true_title'], 'pred_skills_model': pred,
                 'conf': round(float(conf), 3), 'top3': ', '.join(top3),
                 'in_coverage': in_cov, 'is_none': is_none, 'correct': correct,
                 'n_skills': len(cv32_skills[f])})
res32 = pd.DataFrame(rows)
in_cov = res32[res32.correct.notna()]
print(f"in-coverage engineering CVs: {len(in_cov)} | correct: {int(in_cov.correct.sum())} "
      f"({in_cov.correct.mean():.1%})")
print(f"out-of-coverage engineering: {int(((~res32.in_coverage) & (~res32.is_none)).sum())} "
      f"| 'none' fixtures: {int(res32.is_none.sum())}")
res32[['file', 'true', 'pred_skills_model', 'conf', 'in_coverage', 'correct']]

in-coverage engineering CVs: 20 | correct: 15 (75.0%)
out-of-coverage engineering: 7 | 'none' fixtures: 5


,file,true,pred_skills_model,conf,in_coverage,correct
0,backend-senior-strong_Daniel-Peretz.pdf,Backend Developer,UX Designer,0.107,True,False
1,frontend-mid-strong_Noa-Shapiro.pdf,Frontend Developer,Frontend Developer,0.997,True,True
2,devops-senior-strong_Alex-Vaisman.pdf,DevOps Engineer,DevOps Engineer,1.000,True,True
3,datascientist-mid-strong_Maya-Berkovich.pdf,Data Scientist,Data Scientist,0.975,True,True
4,qa-automation-mid-mid_Yossi-Alfasi.pdf,QA Automation Engineer,QA Automation Engineer,1.000,True,True
5,ml-senior-strong_Amir-Dahan.pdf,Machine Learning Engineer,Data Scientist,0.909,False,None
6,java-senior-mid_Marina-Feldman.pdf,Java Developer,Java Developer,1.000,True,True
7,software-mid-mid_Tomer-Azulay.pdf,Software Engineer,Backend Developer,0.995,True,False
8,dataeng-mid-strong_Shira-Golan.pdf,Data Engineer,Data Engineer,0.999,True,True
9,fullstack-mid-ambiguous_Omri-Katz.pdf,Fullstack Engineer,Backend Developer,0.739,True,True


## 5. Head-to-head vs the shipped text classifier

The competitor: `text_to_job_title_classifier.joblib` (raw-text TF-IDF+LogReg,
59 titles + `__other__`). Loaded directly and scored with the same
renormalization the server applies (mirror of `server.py:437-476` —
`__other__` is dropped from the ranking but its probability mass keeps deflating
the shares; mass > 0.5 acts as the 'not an engineering CV' veto).

In [10]:
import joblib
from taxonomy import OTHER_LABEL
text_clf = joblib.load('text_to_job_title_classifier.joblib')

def text_model_predict(text):
    proba = text_clf.predict_proba([text])[0]
    labels = list(text_clf.classes_)
    other_mass = float(proba[labels.index(OTHER_LABEL)]) if OTHER_LABEL in labels else 0.0
    ranked = sorted(zip(labels, proba), key=lambda lp: -lp[1])
    ranked = [(l, p) for l, p in ranked if str(l) != OTHER_LABEL][:3]
    total = sum(p for _, p in ranked) + other_mass
    top, share = ranked[0][0], ranked[0][1] / total * 100
    return str(top), share, other_mass

h2h = []
for entry in manifest:
    f = entry['file']
    acceptable = set(entry['acceptable_titles']) if entry['acceptable_titles'] else set()
    is_none = entry['true_title'] == 'none'
    t_pred, t_share, t_other = text_model_predict(texts[f])
    text_says_none = t_other > 0.5
    if is_none:
        t_correct = text_says_none
    else:
        t_correct = (t_pred in acceptable) and not text_says_none
    s_row = res32[res32.file == f].iloc[0]
    h2h.append({'file': f, 'true': entry['true_title'],
                'skills_pred': s_row.pred_skills_model, 'skills_correct': s_row.correct,
                'text_pred': t_pred if not text_says_none else '(none)',
                'text_correct': t_correct, 'in_coverage': s_row.in_coverage})
h2h = pd.DataFrame(h2h)
fair = h2h[h2h.skills_correct.notna()]
print('=== head-to-head on the fair (in-coverage) set ===')
print(f"skills-model: {int(fair.skills_correct.sum())}/{len(fair)} ({fair.skills_correct.mean():.1%})")
print(f"text-model:   {int(fair.text_correct.sum())}/{len(fair)} ({fair.text_correct.mean():.1%})")
print('\n=== full 32 (text model plays everywhere; skills model only in coverage) ===')
print(f"text-model overall: {int(h2h.text_correct.sum())}/{len(h2h)}")
h2h

=== head-to-head on the fair (in-coverage) set ===
skills-model: 15/20 (75.0%)
text-model:   16/20 (80.0%)

=== full 32 (text model plays everywhere; skills model only in coverage) ===
text-model overall: 16/32


,file,true,skills_pred,skills_correct,text_pred,text_correct,in_coverage
0,backend-senior-strong_Daniel-Peretz.pdf,Backend Developer,UX Designer,False,Java Developer,False,True
1,frontend-mid-strong_Noa-Shapiro.pdf,Frontend Developer,Frontend Developer,True,Frontend Developer,True,True
2,devops-senior-strong_Alex-Vaisman.pdf,DevOps Engineer,DevOps Engineer,True,DevOps Engineer,True,True
3,datascientist-mid-strong_Maya-Berkovich.pdf,Data Scientist,Data Scientist,True,Data Scientist,True,True
4,qa-automation-mid-mid_Yossi-Alfasi.pdf,QA Automation Engineer,QA Automation Engineer,True,QA Automation Engineer,True,True
5,ml-senior-strong_Amir-Dahan.pdf,Machine Learning Engineer,Data Scientist,None,Data Scientist,False,False
6,java-senior-mid_Marina-Feldman.pdf,Java Developer,Java Developer,True,Java Developer,True,True
7,software-mid-mid_Tomer-Azulay.pdf,Software Engineer,Backend Developer,False,Software Engineer,True,True
8,dataeng-mid-strong_Shira-Golan.pdf,Data Engineer,Data Engineer,True,Data Engineer,True,True
9,fullstack-mid-ambiguous_Omri-Katz.pdf,Fullstack Engineer,Backend Developer,True,Frontend Developer,True,True


In [11]:
# Agreement analysis - does disagreement predict error? (the cross-validation claim)
both = fair.copy()
both['agree'] = both.skills_pred == both.text_pred.str.replace('(none)', '', regex=False)
agree = both[both.agree]; disagree = both[~both.agree]
def acc(s): return s.mean() if len(s) else float('nan')
print(f"agree on {len(agree)}/{len(both)} in-coverage CVs")
print(f"  when they AGREE:    text-model correct {acc(agree.text_correct):.1%} | skills {acc(agree.skills_correct):.1%}")
print(f"  when they DISAGREE: text-model correct {acc(disagree.text_correct):.1%} | skills {acc(disagree.skills_correct):.1%}")
print('\nDisagreement rows:')
disagree[['file', 'true', 'skills_pred', 'text_pred', 'skills_correct', 'text_correct']]

agree on 15/20 in-coverage CVs
  when they AGREE:    text-model correct 86.7% | skills 86.7%
  when they DISAGREE: text-model correct 60.0% | skills 40.0%

Disagreement rows:


,file,true,skills_pred,text_pred,skills_correct,text_correct
0,backend-senior-strong_Daniel-Peretz.pdf,Backend Developer,UX Designer,Java Developer,False,False
7,software-mid-mid_Tomer-Azulay.pdf,Software Engineer,Backend Developer,Software Engineer,False,True
9,fullstack-mid-ambiguous_Omri-Katz.pdf,Fullstack Engineer,Backend Developer,Frontend Developer,True,True
12,datasci-ml-mid-ambiguous_Yael-Rosen.pdf,Data Scientist,Software Engineer,Data Scientist,False,True
19,student-junior-weak_Itay-Cohen.pdf,Software Engineer,Software Engineer,Java Developer,True,False


In [12]:
# Persist the research artifact (dated, gitignored pattern) - wiring is a separate decision
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
art_path = f'skills_to_title_{stamp}.joblib'
joblib.dump({'model': primary, 'vectorizer': pvec, 'label_encoder': ple,
             'labels': sorted(COVERED),
             'trained_on': 'lang-uk-cv-skills', 'leak_guard': sorted(LEAK)[:5] + ['...'],
             'trained_at': stamp}, art_path)
print('saved research artifact:', art_path)

saved research artifact: skills_to_title_20260729_115813.joblib


## 6. Wrap-up

Numbers above feed `docs/final-sprint/outputs/18-skills-to-title-results.md`
(kept in sync with this notebook) — including the held-out metrics per domain,
the domain-shift transfer, the 32-CV pipeline test, the head-to-head, and the
agreement analysis. The wiring decision memo lives in the same results file;
wiring into the production detection ladder, if ever, is its own task with its
own kickoff.